In [1]:
import pandas as pd                                  # pandas: tables
from scipy import stats                              # scipy.stats: the t-test
from statsmodels.stats.power import TTestIndPower    # sample-size calculator for number outcomes

In [2]:
url = "https://raw.githubusercontent.com/assemmkhalil/MarketingABtest/main/Marketing-Campaign.csv"
df = pd.read_csv(url)        # read the file into a table called df
print(df.shape)              # 548 rows (137 restaurants x 4 weeks), 7 columns

(548, 7)


<b>In this notebook i would like to work on two promotions only

In [3]:
two_promo = df.loc[(df["Promotion"]==1) | (df["Promotion"]==2), :]

In [4]:
two_promo.shape

(360, 7)

In [5]:
two_promo.isna().sum()

MarketID            0
MarketSize          0
LocationID          0
AgeOfStore          0
Promotion           0
week                0
SalesInThousands    0
dtype: int64

In [6]:
two_promo["SalesInThousands"].describe()

count    360.000000
mean      52.474889
std       16.686955
min       17.340000
25%       41.507500
50%       49.500000
75%       59.672500
max       99.650000
Name: SalesInThousands, dtype: float64

In [7]:
two_promo.head()

,MarketID,MarketSize,LocationID,AgeOfStore,Promotion,week,SalesInThousands
4,1,Medium,2,5,2,1,27.81
5,1,Medium,2,5,2,2,34.67
6,1,Medium,2,5,2,3,27.98
7,1,Medium,2,5,2,4,27.72
8,1,Medium,3,12,1,1,44.54


In [13]:
promotions_per_restaurant=two_promo.groupby("LocationID")["Promotion"].nunique()
print(promotions_per_restaurant.max())

1


In [16]:
promotions_per_restaurant.value_counts()

Promotion
1    90
Name: count, dtype: int64

In [30]:
restaurants = two_promo.groupby(["LocationID", "Promotion", "MarketSize"])["SalesInThousands"].mean().reset_index()
# one per restaurant

The question and the hypotheses <br>
Question: do promotions 1 and 2 give different average weekly sales per restaurant? <br>
• Null hypothesis: both promotions give the same average sales. Any difference is chance. <br>
• Alternative hypothesis: they give different average sales.

Was the sample big enough?
1.  Spread: average weekly sales differ from restaurant to restaurant by about 16 thousand dollars (the standard deviation).
2.  Smallest change worth catching: the company says 10 thousand dollars per week.
3.  Effect size: 10 ÷ 16 = 0.625.
4.  The 0.05 line and power 80%.

In [24]:
needed = TTestIndPower().solve_power(effect_size=10/16, alpha=0.05, power=0.8)
print(needed)    # restaurants needed for each promotion

41.168919258185596


In [39]:
counts=restaurants.Promotion.value_counts()

In [40]:
counts/counts.sum() * 100

Promotion
2    52.222222
1    47.777778
Name: count, dtype: float64

In [42]:
print(stats.chisquare(counts).pvalue)

0.6732899796599957


Above 0.05, so the 43/47 split is normal chance.

In [ ]:
# area size in each group

print(pd.crosstab(restaurants["Promotion"], restaurants["MarketSize"]))

MarketSize  Large  Medium  Small
Promotion                       
1              14      24      5
2              16      27      4


The two groups have a similar mix of small, medium and large areas, so area size cannot explain a difference in sales

 Did the effect fade over the weeks? (novelty effect)

In [46]:
weekly = pd.crosstab(two_promo["week"],two_promo["Promotion"], values=two_promo["SalesInThousands"], aggfunc="mean")
print(weekly.round(1))

Promotion     1     2
week                 
1          58.2  47.7
2          56.9  47.6
3          58.8  47.7
4          58.4  46.3


In [47]:
two_promo

,MarketID,MarketSize,LocationID,AgeOfStore,Promotion,week,SalesInThousands
4,1,Medium,2,5,2,1,27.81
5,1,Medium,2,5,2,2,34.67
6,1,Medium,2,5,2,3,27.98
7,1,Medium,2,5,2,4,27.72
8,1,Medium,3,12,1,1,44.54
...,...,...,...,...,...,...,...
543,10,Large,919,2,1,4,64.34
544,10,Large,920,14,2,1,50.20
545,10,Large,920,14,2,2,45.75
546,10,Large,920,14,2,3,44.29


# The main test

In [48]:
promo1_sales=restaurants.loc[restaurants["Promotion"]==1,"SalesInThousands"]
promo2_sales=restaurants.loc[restaurants["Promotion"]==2,"SalesInThousands"]

In [51]:
print(promo1_sales.mean().round(1),promo2_sales.mean().round(1))

58.1 47.3


In [52]:
print(stats.ttest_ind(promo1_sales,promo2_sales).pvalue)

0.0012025674613766499


How big is the effect? (Cohen's d)

In [53]:
gap = promo1_sales.mean() - promo2_sales.mean()                              # 58.1 - 47.3 = 10.8
combined_sd = ((promo1_sales.std()**2 + promo2_sales.std()**2) / 2) ** 0.5    # typical spread of the two groups together
print(gap / combined_sd)                                            # Cohen's d

0.7047313069495937


 ### Decision
Report sentence: "Restaurants with promotion 1 sold on average 58.1 thousand dollars per week, restaurants with promotion 2 sold 47.3 (t-test p = 0.0012, Cohen's d = 0.70). The difference was stable over the 4 weeks."
Decision: use promotion 1, not promotion 2.